# Positive-Control Responses Merge (n=30, six methods)

Merges the ALREADY-GENERATED positive-control responses from all six benchmark
arms into a single Excel sheet: one row per question, one column per model,
each cell holding that model's full response text.

No model is called here - this notebook only reads the three result files the
generation notebooks already wrote:

- notebook 3: output/ScientificReportRevisionV1/batch_eval/responses_latest.csv
  (T21 RAG, GPT-5.4, Claude, Nemotron)
- notebook 5: output/medrag_benchmark/medrag_benchmark_results.csv (MedRAG)
- notebook 6: output/paperqa_benchmark/paperqa_benchmark_results.csv (PaperQA2)

Alignment and cleaning follow notebook 7 exactly: the three files are asserted
to hold the same 30 questions in the same order (whitespace/case-normalised),
residual PMC citation tokens are stripped from the T21 answers, and PaperQA2's
cleaned text (paperqa_clean) is preferred when present.

Output: one workbook with a single All_responses sheet (30 question rows x
6 model columns, plus sl_no, question and ground truth), and a CSV copy.

## Cell 1 - Load and align the three result files

In [1]:
# ---- Cell 1: load and align (mirrors notebook 7 Cell 3) ----
import re
from pathlib import Path

import pandas as pd

ANALYSIS = Path('..').resolve()
RESP_CSV    = ANALYSIS / 'output/ScientificReportRevisionV1/batch_eval/responses_latest.csv'
MEDRAG_CSV  = ANALYSIS / 'output/medrag_benchmark/medrag_benchmark_results.csv'
PAPERQA_CSV = ANALYSIS / 'output/paperqa_benchmark/paperqa_benchmark_results.csv'
OUT_DIR     = ANALYSIS / 'output/ScientificReportRevisionV1/positive_control_responses'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# (key, label) - same order as notebook 7
MODELS = [
    ('t21',      'T21 RAG'),
    ('nemotron', 'Nemotron'),
    ('gpt',      'GPT-5.4'),
    ('claude',   'Claude'),
    ('medrag',   'MedRAG'),
    ('paperqa',  'PaperQA2'),
]

_PMC_TOK_RE = re.compile(r'\s*\[{1,2}PMC\d{4,}[^\]]*\]{1,2}(?:\([^)]*\))?')
_norm = lambda s: re.sub(r'\s+', ' ', str(s).strip().lower())

a = pd.read_csv(RESP_CSV)
m = pd.read_csv(MEDRAG_CSV)
p = pd.read_csv(PAPERQA_CSV)
assert len(a) == len(m) == len(p) == 30, f'expected 30 rows each, got {len(a)}/{len(m)}/{len(p)}'
assert (a['question'].map(_norm).values == m['question'].map(_norm).values).all(), \
    'question order mismatch: responses_latest vs medrag'
assert (a['question'].map(_norm).values == p['question'].map(_norm).values).all(), \
    'question order mismatch: responses_latest vs paperqa'
assert (a['ground_truth'].map(_norm).values == m['ground_truth'].map(_norm).values).all(), \
    'ground-truth mismatch: responses_latest vs medrag'

answers = {
    't21':      a['t21_answer'].fillna('').astype(str).apply(
                    lambda x: re.sub(r'\s{2,}', ' ', _PMC_TOK_RE.sub('', x)).strip()),
    'gpt':      a['gpt_answer'].fillna('').astype(str),
    'claude':   a['claude_answer'].fillna('').astype(str),
    'nemotron': a['nemotron_answer'].fillna('').astype(str),
    'medrag':   m['medrag_answer'].fillna('').astype(str),
    'paperqa':  (p['paperqa_clean'] if 'paperqa_clean' in p.columns
                 else p['paperqa_answer']).fillna('').astype(str),
}
print(f'{len(a)} aligned positive-control questions, 6 arms')
for k, lbl in MODELS:
    print(f'  {lbl:9s} mean words: {answers[k].str.split().str.len().mean():5.0f}')


30 aligned positive-control questions, 6 arms
  T21 RAG   mean words:    95
  Nemotron  mean words:    98
  GPT-5.4   mean words:   125
  Claude    mean words:   134
  MedRAG    mean words:   280
  PaperQA2  mean words:   133


## Cell 2 - Merge into one Excel sheet

One row per positive-control question, one column per model, each cell the
full response. Ground truth is kept as a reference column next to the
question. Saved as positive_control_responses_merged.xlsx (single
All_responses sheet) plus a CSV copy.

In [2]:
# ---- Cell 2: merge and save ----
df_all = pd.DataFrame({
    'sl_no':        a['sl_no'],
    'question':     a['question'],
    'ground_truth': a['ground_truth'].fillna('').astype(str),
})
for k, lbl in MODELS:
    df_all[lbl] = answers[k].values

RESULTS_XLSX = OUT_DIR / 'positive_control_responses_merged.xlsx'
with pd.ExcelWriter(RESULTS_XLSX, engine='openpyxl') as xw:
    df_all.to_excel(xw, sheet_name='All_responses', index=False)
    # Widen columns for readability (engine pinned: xlsxwriter is also
    # installed in dschat and would otherwise be picked, breaking the
    # openpyxl column_dimensions API below)
    from openpyxl.utils import get_column_letter
    ws = xw.sheets['All_responses']
    ws.column_dimensions['B'].width = 45   # question
    ws.column_dimensions['C'].width = 50   # ground truth
    for _ci in range(4, 4 + len(MODELS)):  # model columns D..I
        ws.column_dimensions[get_column_letter(_ci)].width = 60

df_all.to_csv(OUT_DIR / 'positive_control_responses_merged.csv',
              index=False, encoding='utf-8-sig')
print(f'saved -> {RESULTS_XLSX}')
print(f'saved -> {RESULTS_XLSX.with_suffix(".csv").name}')
print(f'shape  : {df_all.shape[0]} questions x {len(MODELS)} model columns')
df_all.head(3)


saved -> /path/to/project/T21_research_assistant_v7/analysis/output/ScientificReportRevisionV1/positive_control_responses/positive_control_responses_merged.xlsx
saved -> positive_control_responses_merged.csv
shape  : 30 questions x 6 model columns


,sl_no,question,ground_truth,T21 RAG,Nemotron,GPT-5.4,Claude,MedRAG,PaperQA2
0,1,Which clinical conditions are people with Down...,Individuals with Down syndrome are at high ris...,Trisomy 21 creates a broad spectrum of health ...,Individuals with Down syndrome have a markedly...,People with Down syndrome have elevated lifeti...,People with Down syndrome face significantly e...,People with Down syndrome are at significantly...,People with Down syndrome have markedly increa...
1,2,Which clinical conditions are people with Down...,Individuals with Down syndrome are at low risk...,Individuals with Down syndrome (DS) exhibit a ...,Individuals with Down syndrome have a markedly...,"Compared with the general population, people w...",People with Down syndrome exhibit a notably re...,People with Down syndrome appear to have a **l...,Individuals with Down syndrome have a signific...
2,3,Why do people with Down syndrome tend to have ...,"Four interferon receptors, IFNAR1, IFNAR2, IFN...",Trisomy 21 (T21) leads to overexpression of im...,Trisomy 21 causes a dosage imbalance of chromo...,People with Down syndrome often show chronic i...,People with Down syndrome exhibit chronic immu...,People with Down syndrome (DS) show a characte...,Trisomy 21 leads to an extra copy of chromosom...


---

Output written to
analysis/output/ScientificReportRevisionV1/positive_control_responses/:

- positive_control_responses_merged.xlsx - single All_responses sheet, 30
  question rows x 6 model columns (T21 RAG, Nemotron, GPT-5.4, Claude,
  MedRAG, PaperQA2), each cell one full response; sl_no, question and
  ground truth as the leading columns
- positive_control_responses_merged.csv - same table as CSV